# Task 14: Thinning and Skeletonization

Implement Zhang-Suen thinning and morphological skeletonization manually with NumPy. These methods reduce binary objects to thin structural representations.

## 1. Introduction

Thinning iteratively removes boundary pixels while trying to preserve connectivity. Morphological skeletonization accumulates layers from erosions and openings. They are related but different methods, so their outputs may differ.

## 2. Libraries and Binary Images

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

In [ ]:
vertical_bar = np.zeros((21, 21), dtype=np.uint8)
vertical_bar[3:18, 7:14] = 255
thick_cross = np.zeros((21, 21), dtype=np.uint8)
thick_cross[8:13, 3:18] = 255
thick_cross[3:18, 8:13] = 255
letter_t = np.zeros((21, 21), dtype=np.uint8)
letter_t[3:8, 3:18] = 255
letter_t[3:18, 8:13] = 255

fig, axes = plt.subplots(1, 3, figsize=(9, 3))
for axis, image, title in zip(
    axes,
    (vertical_bar, thick_cross, letter_t),
    ("Thick bar", "Thick cross", "Letter T"),
    strict=True,
):
    axis.imshow(image, cmap="gray", vmin=0, vmax=255)
    axis.set_title(title)
    axis.axis("off")

## 3. Pixel Neighborhoods

Zhang-Suen uses the circular sequence `P2, P3, P4, P5, P6, P7, P8, P9` around the center pixel `P1`:

```text
P9 P2 P3
P8 P1 P4
P7 P6 P5
```

The number of 0-to-1 transitions is counted around this circular sequence, including the transition from `P9` back to `P2`.

## 4. Zhang-Suen Conditions

For a foreground pixel, let `B` be its number of foreground neighbors and `A` its circular 0-to-1 transitions. In both steps, require `2 <= B <= 6` and `A == 1`.

- Step 1: `P2 * P4 * P6 == 0` and `P4 * P6 * P8 == 0`.
- Step 2: `P2 * P4 * P8 == 0` and `P2 * P6 * P8 == 0`.

Mark all eligible pixels first, then remove them simultaneously.

## 5. Neighborhood Transitions

In [ ]:
def contar_transicoes_01(vizinhos):
    """Counts circular 0-to-1 transitions in P2, P3, ..., P9."""

    ### START CODE HERE ###

    # TODO: Count circular 0-to-1 transitions.
    raise NotImplementedError("Implement contar_transicoes_01.")

    ### END CODE HERE ###

## 6. Thinning Step

In [ ]:
def passo_afinamento(imagem, etapa):
    """Runs one Zhang-Suen subiteration, where etapa is 1 or 2."""

    ### START CODE HERE ###

    # TODO: Implement one Zhang-Suen subiteration.
    raise NotImplementedError("Implement passo_afinamento.")

    ### END CODE HERE ###

## 7. Zhang-Suen Thinning

In [ ]:
def afinamento_zhang_suen(imagem):
    """Applies Zhang-Suen subiterations until no pixel is removed."""

    ### START CODE HERE ###

    # TODO: Repeatedly apply both thinning steps until convergence.
    raise NotImplementedError("Implement afinamento_zhang_suen.")

    ### END CODE HERE ###

## 8. Morphological Skeletonization

The helper morphology functions below are provided and already follow the zero-background boundary convention. For each erosion, accumulate the eroded image minus its opening until the image becomes empty.

In [ ]:
def _validate_binary(image):
    image = np.asarray(image)
    if image.ndim != 2 or not np.all(np.isin(image, [0, 255])):
        raise ValueError("image must be a 2D binary array with values 0 or 255")
    return image


def erosao_fornecida(image, element):
    """Provided binary erosion with background outside the image."""
    image = _validate_binary(image)
    element = np.asarray(element, dtype=bool)
    if element.ndim != 2 or any(size % 2 == 0 for size in element.shape):
        raise ValueError("element must have odd 2D dimensions")
    rows, cols = image.shape
    row_anchor, col_anchor = np.array(element.shape) // 2
    result = np.zeros_like(image)
    for row in range(rows):
        for col in range(cols):
            keep = True
            for element_row, element_col in zip(*np.nonzero(element), strict=True):
                source_row = row + element_row - row_anchor
                source_col = col + element_col - col_anchor
                if (
                    not (0 <= source_row < rows and 0 <= source_col < cols)
                    or image[source_row, source_col] == 0
                ):
                    keep = False
                    break
            if keep:
                result[row, col] = 255
    return result


def dilatacao_fornecida(image, element):
    """Provided binary dilation with background outside the image."""
    image = _validate_binary(image)
    element = np.asarray(element, dtype=bool)
    rows, cols = image.shape
    row_anchor, col_anchor = np.array(element.shape) // 2
    result = np.zeros_like(image)
    for row in range(rows):
        for col in range(cols):
            for element_row, element_col in zip(*np.nonzero(element), strict=True):
                source_row = row + element_row - row_anchor
                source_col = col + element_col - col_anchor
                if (
                    0 <= source_row < rows
                    and 0 <= source_col < cols
                    and image[source_row, source_col] == 255
                ):
                    result[row, col] = 255
                    break
    return result


def abertura_fornecida(image, element):
    """Provided opening for use in morphological skeletonization."""
    return dilatacao_fornecida(erosao_fornecida(image, element), element)

In [ ]:
def esqueletonizacao_morfologica(imagem, elemento_estruturante):
    """Builds a morphological skeleton from successive erosions and openings."""

    ### START CODE HERE ###

    # TODO: Implement morphological skeletonization.
    raise NotImplementedError("Implement esqueletonizacao_morfologica.")

    ### END CODE HERE ###

## 9. Deterministic Tests

Run these tests only after completing every TODO. They verify transitions, convergence, idempotence, boundaries, binary outputs, input preservation, and the expected 5x5-square skeleton.

In [ ]:
def assert_binary(output, shape):
    assert output.shape == shape
    assert set(np.unique(output)).issubset({0, 255})


assert contar_transicoes_01(np.array([0, 1, 1, 0, 0, 1, 0, 0])) == 2
assert contar_transicoes_01(np.array([1, 0, 0, 0, 0, 0, 0, 0])) == 1
assert contar_transicoes_01(np.zeros(8, dtype=np.uint8)) == 0

line = np.zeros((9, 9), dtype=np.uint8)
line[4, 2:7] = 255
np.testing.assert_array_equal(afinamento_zhang_suen(line), line)

thick_object = np.zeros((11, 11), dtype=np.uint8)
thick_object[2:9, 3:8] = 255
thinned = afinamento_zhang_suen(thick_object)
assert np.count_nonzero(thinned) < np.count_nonzero(thick_object)
assert np.all(thinned[thick_object == 0] == 0)
np.testing.assert_array_equal(afinamento_zhang_suen(thinned), thinned)

square = np.zeros((9, 9), dtype=np.uint8)
square[2:7, 2:7] = 255
square_before = square.copy()
square_element = np.ones((3, 3), dtype=np.uint8)
skeleton = esqueletonizacao_morfologica(square, square_element)
expected_skeleton = np.zeros((9, 9), dtype=np.uint8)
expected_skeleton[4, 4] = 255
np.testing.assert_array_equal(skeleton, expected_skeleton)
np.testing.assert_array_equal(square, square_before)

empty = np.zeros((7, 5), dtype=np.uint8)
np.testing.assert_array_equal(
    esqueletonizacao_morfologica(empty, square_element), empty
)

for operation, args in (
    (passo_afinamento, (thick_object, 1)),
    (afinamento_zhang_suen, (thick_object,)),
    (esqueletonizacao_morfologica, (thick_object, square_element)),
):
    result = operation(*args)
    assert_binary(result, thick_object.shape)

print("Test passed!")

## 10. Visual Comparison

After your tests pass, compare both methods on several objects. The algorithms may return different structures.

In [ ]:
examples = (
    ("Thick bar", vertical_bar),
    ("Thick cross", thick_cross),
    ("Letter T", letter_t),
)
fig, axes = plt.subplots(len(examples), 3, figsize=(9, 9))
for row, (name, image) in enumerate(examples):
    outputs = (
        ("Original", image),
        ("Zhang-Suen", afinamento_zhang_suen(image)),
        (
            "Morphological skeleton",
            esqueletonizacao_morfologica(image, np.ones((3, 3), dtype=np.uint8)),
        ),
    )
    for axis, (title, output) in zip(axes[row], outputs, strict=True):
        axis.imshow(output, cmap="gray", vmin=0, vmax=255)
        axis.set_title(f"{name}: {title}")
        axis.axis("off")
plt.tight_layout()

## 11. Results and Discussion

1. What is the difference between thinning and morphological skeletonization?
2. Why is preserving connectivity important during thinning?
3. Why does Zhang-Suen use two subiterations?
4. How does the structuring element affect morphological skeletonization?
5. Why might the two algorithms produce different results?
6. In which computer vision applications can thinning and skeletonization be useful?